# Digital Inspector — family & stage classifier training

Trains the two mmBERT-small classifiers from CLAUDE.md §5, calibrates the family model, exports both to ONNX INT8, and builds the FAISS similarity index over `scripts_index.jsonl`.

**Kaggle setup before running:**
1. Settings → Accelerator → GPU (T4 x2 or P100).
2. Settings → Internet → On (needed to pull base models from the Hub).
3. Add Data → upload `train.jsonl`, `eval.jsonl`, `stage_labels.jsonl`, `scripts_index.jsonl` from `data/processed/` as a Kaggle Dataset, or point `DATA_DIR` below at wherever you've attached them.

Outputs land in `/kaggle/working/models/` — Kaggle keeps this when you "Save Version", which is this notebook's equivalent of "checkpoint to Drive, not local disk".

In [ ]:
!pip install "optimum[onnxruntime]" onnx onnxruntime

In [4]:
!pip install -q -U "transformers>=4.48"
!pip install -q faiss-cpu sentencepiece accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 108.1 MB/s eta 0:00:0000:0100:01


In [41]:
import os
from pathlib import Path

print("cwd:", os.getcwd())
print("files in /content:", [p.name for p in Path("/content").glob("*.jsonl")])
print("files in /content/models:", [p.name for p in Path("/content/models").glob("*")])
print("train exists:", Path("/content/train.jsonl").exists())
print("eval exists:", Path("/content/eval.jsonl").exists())
print("stage exists:", Path("/content/stage_labels.jsonl").exists())
print("scripts exists:", Path("/content/scripts_index.jsonl").exists())

cwd: /content
files in /content: ['stage_labels.jsonl', 'eval.jsonl', 'train.jsonl', 'scripts_index.jsonl']
files in /content/models: ['config.json', 'tokenizer.json', 'special_tokens_map.json', 'tokenizer_config.json']
train exists: True
eval exists: True
stage exists: True
scripts exists: True


In [1]:
!nvidia-smi

Tue Jul 14 10:45:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [24]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

import json
import subprocess
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset

SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)

DATA_DIR_CANDIDATES = [
    Path.cwd() / "data/processed",
    Path.cwd(),
    Path("/content/drive/MyDrive/digital-inspector/data/processed"),
    Path("/content/drive/MyDrive/data/processed"),
    Path("/content/digital-inspector/data/processed"),
    Path("/content") / "data/processed",
    Path("/content"),
    Path("/content/sample_data"),
    Path("/kaggle/input/digital-inspector/data/processed"),
    Path("../data/processed"),
    Path("data/processed"),
]
REQUIRED_FILES = ["train.jsonl", "eval.jsonl", "stage_labels.jsonl", "scripts_index.jsonl"]
DATA_DIR = None
for candidate in DATA_DIR_CANDIDATES:
    if all((candidate / name).exists() for name in REQUIRED_FILES):
        DATA_DIR = candidate
        break
if DATA_DIR is None:
    searched = [str(path) for path in DATA_DIR_CANDIDATES]
    raise FileNotFoundError(f"Could not find the processed data folder. Upload the four JSONL files into one folder and set DATA_DIR to it. Searched: {searched}")

OUTPUT_DIR = Path("/kaggle/working/models")
if not Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR={DATA_DIR} (exists={DATA_DIR.exists()})")
print(f"OUTPUT_DIR={OUTPUT_DIR}")

DATA_DIR=/content (exists=True)
OUTPUT_DIR=models


## Base model canary

`jhu-clsp/mmBERT-small` needs `transformers>=4.48` for ModernBERT architecture support. Verify it loads *before* anything else — if it fails, fall back immediately rather than debugging the base model choice.

In [38]:
from transformers import AutoTokenizer, __version__ as tf_version

print(f"transformers version: {tf_version}")

BASE_MODEL_CANDIDATES = [
    Path("/content/models/mmBERT-small"),
    Path("/content/models"),
]
BASE_MODEL_PATH = next((path for path in BASE_MODEL_CANDIDATES if (path / "config.json").exists()), None)
if BASE_MODEL_PATH is None:
    searched = [str(path) for path in BASE_MODEL_CANDIDATES]
    raise FileNotFoundError(f"Local base model folder not found. Searched: {searched}")
BASE_MODEL = str(BASE_MODEL_PATH)

print(f"Using local base model folder: {BASE_MODEL}")

transformers version: 5.13.1
Using local base model folder: /content/models


## Load the frozen split

`train.jsonl` / `eval.jsonl` are the family-classifier split from `data/05_split.py` — eval is real/real-derived call-shaped content only (email-sourced `fredzhang7` rows are excluded from eval and capped in train, see `data/README.md`).

In [ ]:
FAMILY_IDS = [
    "digital_arrest",
    "kyc_bank_fraud",
    "parcel_courier",
    "tech_support",
    "refund_reward",
    "investment_fraud",
    "legitimate",
]
FAMILY_LABEL2ID = {label: i for i, label in enumerate(FAMILY_IDS)}
FAMILY_ID2LABEL = {i: label for i, label in enumerate(FAMILY_IDS)}

STAGE_IDS = [
    "s0_none",
    "s1_authority_claim",
    "s2_threat_urgency",
    "s3_isolation",
    "s4_info_harvest",
    "s5_payment_demand",
]
STAGE_LABEL2ID = {label: i for i, label in enumerate(STAGE_IDS)}
STAGE_ID2LABEL = {i: label for i, label in enumerate(STAGE_IDS)}


def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def full_transcript(row):
    return "\n".join(t["text"] for t in row["turns"])


family_train_raw = load_jsonl(DATA_DIR / "train.jsonl")
family_eval_raw = load_jsonl(DATA_DIR / "eval.jsonl")
print(f"family train: {len(family_train_raw)} rows, eval: {len(family_eval_raw)} rows")

In [27]:
family_train_ds = Dataset.from_dict({
    "text": [full_transcript(r) for r in family_train_raw],
    "label": [FAMILY_LABEL2ID[r["family"]] for r in family_train_raw],
})
family_eval_ds = Dataset.from_dict({
    "text": [full_transcript(r) for r in family_eval_raw],
    "label": [FAMILY_LABEL2ID[r["family"]] for r in family_eval_raw],
})
print(family_train_ds)
print(family_eval_ds)

Dataset({
    features: ['text', 'label'],
    num_rows: 7540
})
Dataset({
    features: ['text', 'label'],
    num_rows: 216
})


## Run A — family classifier

Full transcript, max_length 2048, 7 labels, class-weighted loss, lr 2e-5, up to 4 epochs with early stop on eval macro-F1.

In [28]:
from collections import Counter

family_counts = Counter(family_train_ds["label"])
family_total = sum(family_counts.values())
family_class_weights = torch.tensor(
    [family_total / (len(FAMILY_IDS) * family_counts.get(i, 1)) for i in range(len(FAMILY_IDS))],
    dtype=torch.float,
)
print("family class weights:", dict(zip(FAMILY_IDS, family_class_weights.tolist())))

family class weights: {'digital_arrest': 2.69960618019104, 'kyc_bank_fraud': 1.4458293914794922, 'parcel_courier': 11.708074569702148, 'tech_support': 1.8539464473724365, 'refund_reward': 0.5809832215309143, 'investment_fraud': 2.919086217880249, 'legitimate': 0.30775511264801025}


In [29]:
from transformers import AutoTokenizer

family_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
FAMILY_MAX_LENGTH = 2048


def tokenize_family(batch):
    return family_tokenizer(batch["text"], truncation=True, max_length=FAMILY_MAX_LENGTH, padding="max_length")


family_train_tok = family_train_ds.map(tokenize_family, batched=True, remove_columns=["text"])
family_eval_tok = family_eval_ds.map(tokenize_family, batched=True, remove_columns=["text"])

Map:   0%|          | 0/7540 [00:00<?, ? examples/s]

Map:   0%|          | 0/216 [00:00<?, ? examples/s]

In [30]:
from transformers import AutoModelForSequenceClassification

family_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(FAMILY_IDS),
    id2label=FAMILY_ID2LABEL,
    label2id=FAMILY_LABEL2ID,
)

pytorch_model.bin:   0%|          | 0.00/564M [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = torch.nn.CrossEntropyLoss(weight=weight)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


def make_compute_metrics(id_list):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
            "accuracy": (preds == labels).mean(),
        }
    return compute_metrics

In [ ]:
family_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "family_checkpoints"),
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=4,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=25,
    report_to=[],
    seed=SEED,
)

family_trainer = WeightedTrainer(
    model=family_model,
    args=family_args,
    train_dataset=family_train_tok,
    eval_dataset=family_eval_tok,
    class_weights=family_class_weights,
    compute_metrics=make_compute_metrics(FAMILY_IDS),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

family_trainer.train()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

family_eval_output = family_trainer.predict(family_eval_tok)
family_preds = np.argmax(family_eval_output.predictions, axis=-1)
family_labels = family_eval_output.label_ids

print(classification_report(family_labels, family_preds, target_names=FAMILY_IDS, zero_division=0))

family_cm = confusion_matrix(family_labels, family_preds, labels=list(range(len(FAMILY_IDS))))
plt.figure(figsize=(8, 6))
sns.heatmap(family_cm, annot=True, fmt="d", xticklabels=FAMILY_IDS, yticklabels=FAMILY_IDS, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Family classifier — confusion matrix (real eval)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "family_confusion_matrix.png", dpi=150)
plt.show()

family_support = Counter(family_labels)
print("eval support per class:", {FAMILY_IDS[i]: family_support.get(i, 0) for i in range(len(FAMILY_IDS))})

## Calibration

Temperature scaling fit on the eval logits (Guo et al. 2017). Stored as a single scalar `T`; the serving API divides logits by `T` before softmax and reports `calibrated: true`.

In [ ]:
class TemperatureScaler(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = torch.nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature


def fit_temperature(logits, labels):
    scaler = TemperatureScaler()
    logits_t = torch.tensor(logits, dtype=torch.float)
    labels_t = torch.tensor(labels, dtype=torch.long)
    optimizer = torch.optim.LBFGS([scaler.temperature], lr=0.01, max_iter=50)
    nll = torch.nn.CrossEntropyLoss()

    def closure():
        optimizer.zero_grad()
        loss = nll(scaler(logits_t), labels_t)
        loss.backward()
        return loss

    optimizer.step(closure)
    return scaler.temperature.item()


family_temperature = fit_temperature(family_eval_output.predictions, family_labels)
print(f"fitted temperature: {family_temperature:.4f}")

with open(OUTPUT_DIR / "calibration.json", "w") as f:
    json.dump({"family_temperature": family_temperature}, f, indent=2)

## Export family model to ONNX INT8

Exported immediately after training/eval/calibration — while the model is still hot in this session, not deferred to the end.

In [ ]:
family_hf_dir = OUTPUT_DIR / "family_hf"
family_trainer.save_model(str(family_hf_dir))
family_tokenizer.save_pretrained(str(family_hf_dir))

family_onnx_dir = OUTPUT_DIR / "family_onnx"
subprocess.run([
    "optimum-cli", "export", "onnx",
    "--model", str(family_hf_dir),
    "--task", "text-classification",
    str(family_onnx_dir),
], check=True)

In [ ]:
from optimum.onnxruntime import ORTQuantizer, ORTModelForSequenceClassification
from optimum.onnxruntime.configuration import AutoQuantizationConfig

family_ort_model = ORTModelForSequenceClassification.from_pretrained(family_onnx_dir)
family_quantizer = ORTQuantizer.from_pretrained(family_ort_model)
family_qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)

family_int8_dir = OUTPUT_DIR / "family_int8"
family_quantizer.quantize(save_dir=str(family_int8_dir), quantization_config=family_qconfig)
family_tokenizer.save_pretrained(str(family_int8_dir))

print(sorted(p.name for p in family_int8_dir.iterdir()))

## Run B — stage classifier

Utterance-level, max_length 128, 6 labels.

**Data choice.** The stage model classifies one utterance at a time, so it needs clean single-turn examples. Two sources in `stage_labels.jsonl` are unusable for that and are dropped: `kaggle_ieee_scam` rows are multi-stage call *summaries* (a single row spans authority-claim → threat → info-harvest → payment yet carries one arbitrary stage label), and `fredzhang7_all_scam_spam` is email/SMS text. The BothBosu dialogues are the only source of properly segmented conversational turns, so the split is built from them, **held out by `dialogue_id`** so no utterance from an eval conversation leaks into train. The real Indian fraud-call utterances from `kaggle_fraud_call_india` are added to train for in-domain phrasing.

Eval is therefore held-out synthetic-dialogue utterances — the same honesty caveat as the `digital_arrest` family eval (no public per-utterance stage-labelled real Indian dataset exists). This replaces the earlier split, which graded the model on ~34 of those multi-stage summary rows and produced a misleadingly low macro-F1.

In [ ]:
import random

STAGE_SOURCES = {
    "bothbosu_scam_dialogue", "bothbosu_single_agent", "bothbosu_multi_agent",
    "bothbosu_scammer_conversation",
}

stage_rows = load_jsonl(DATA_DIR / "stage_labels.jsonl")
print(f"stage rows total: {len(stage_rows)}")

clean_rows = [r for r in stage_rows if r["source"] in STAGE_SOURCES]
indian_extra = [
    r for r in stage_rows
    if r["source"] == "kaggle_fraud_call_india" and r["stage"] != "s0_none"
]
print(f"clean single-turn utterances: {len(clean_rows)}, real Indian utterances added to train: {len(indian_extra)}")

dialogue_ids = sorted({r["dialogue_id"] for r in clean_rows})
rng = random.Random(SEED)
rng.shuffle(dialogue_ids)
n_eval_dialogues = round(len(dialogue_ids) * 0.18)
eval_dialogue_ids = set(dialogue_ids[:n_eval_dialogues])

stage_eval_raw = [r for r in clean_rows if r["dialogue_id"] in eval_dialogue_ids]
stage_train_raw = [r for r in clean_rows if r["dialogue_id"] not in eval_dialogue_ids] + indian_extra

print(f"stage train: {len(stage_train_raw)}, stage eval: {len(stage_eval_raw)}")
print(f"held-out dialogues: {len(eval_dialogue_ids)} of {len(dialogue_ids)}")
print("eval support per stage:", dict(sorted(Counter(r["stage"] for r in stage_eval_raw).items())))

In [ ]:
stage_train_ds = Dataset.from_dict({
    "text": [r["text"] for r in stage_train_raw],
    "label": [STAGE_LABEL2ID[r["stage"]] for r in stage_train_raw],
})
stage_eval_ds = Dataset.from_dict({
    "text": [r["text"] for r in stage_eval_raw],
    "label": [STAGE_LABEL2ID[r["stage"]] for r in stage_eval_raw],
})

stage_counts = Counter(stage_train_ds["label"])
stage_total = sum(stage_counts.values())
stage_class_weights = torch.tensor(
    [stage_total / (len(STAGE_IDS) * stage_counts.get(i, 1)) for i in range(len(STAGE_IDS))],
    dtype=torch.float,
)
print("stage class weights:", dict(zip(STAGE_IDS, stage_class_weights.tolist())))

In [ ]:
stage_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
STAGE_MAX_LENGTH = 128


def tokenize_stage(batch):
    return stage_tokenizer(batch["text"], truncation=True, max_length=STAGE_MAX_LENGTH, padding="max_length")


stage_train_tok = stage_train_ds.map(tokenize_stage, batched=True, remove_columns=["text"])
stage_eval_tok = stage_eval_ds.map(tokenize_stage, batched=True, remove_columns=["text"])

stage_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(STAGE_IDS),
    id2label=STAGE_ID2LABEL,
    label2id=STAGE_LABEL2ID,
)

In [ ]:
stage_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "stage_checkpoints"),
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=25,
    report_to=[],
    seed=SEED,
)

stage_trainer = WeightedTrainer(
    model=stage_model,
    args=stage_args,
    train_dataset=stage_train_tok,
    eval_dataset=stage_eval_tok,
    class_weights=stage_class_weights,
    compute_metrics=make_compute_metrics(STAGE_IDS),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

stage_trainer.train()

In [ ]:
stage_eval_output = stage_trainer.predict(stage_eval_tok)
stage_preds = np.argmax(stage_eval_output.predictions, axis=-1)
stage_labels_arr = stage_eval_output.label_ids

print(classification_report(stage_labels_arr, stage_preds, target_names=STAGE_IDS, zero_division=0))

stage_cm = confusion_matrix(stage_labels_arr, stage_preds, labels=list(range(len(STAGE_IDS))))
plt.figure(figsize=(8, 6))
sns.heatmap(stage_cm, annot=True, fmt="d", xticklabels=STAGE_IDS, yticklabels=STAGE_IDS, cmap="Oranges")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Stage classifier — confusion matrix (real eval)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "stage_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
stage_hf_dir = OUTPUT_DIR / "stage_hf"
stage_trainer.save_model(str(stage_hf_dir))
stage_tokenizer.save_pretrained(str(stage_hf_dir))

stage_onnx_dir = OUTPUT_DIR / "stage_onnx"
subprocess.run([
    "optimum-cli", "export", "onnx",
    "--model", str(stage_hf_dir),
    "--task", "text-classification",
    str(stage_onnx_dir),
], check=True)

stage_ort_model = ORTModelForSequenceClassification.from_pretrained(stage_onnx_dir)
stage_quantizer = ORTQuantizer.from_pretrained(stage_ort_model)
stage_qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)

stage_int8_dir = OUTPUT_DIR / "stage_int8"
stage_quantizer.quantize(save_dir=str(stage_int8_dir), quantization_config=stage_qconfig)
stage_tokenizer.save_pretrained(str(stage_int8_dir))

print(sorted(p.name for p in stage_int8_dir.iterdir()))

## Embedder + FAISS index

`intfloat/multilingual-e5-small` → ONNX INT8, then a FAISS index over every dialogue in `scripts_index.jsonl` for the closest-known-script feature. E5 convention: index passages with a `"passage: "` prefix; the serving API will prefix queries with `"query: "`.

In [ ]:
E5_MODEL = "intfloat/multilingual-e5-small"
e5_hf_dir = OUTPUT_DIR / "e5_hf"

from transformers import AutoModel

e5_tokenizer = AutoTokenizer.from_pretrained(E5_MODEL)
e5_model = AutoModel.from_pretrained(E5_MODEL)
e5_tokenizer.save_pretrained(str(e5_hf_dir))
e5_model.save_pretrained(str(e5_hf_dir))

e5_onnx_dir = OUTPUT_DIR / "e5_onnx"
subprocess.run([
    "optimum-cli", "export", "onnx",
    "--model", str(e5_hf_dir),
    "--task", "feature-extraction",
    str(e5_onnx_dir),
], check=True)

from optimum.onnxruntime import ORTModelForFeatureExtraction

e5_ort_model = ORTModelForFeatureExtraction.from_pretrained(e5_onnx_dir)
e5_quantizer = ORTQuantizer.from_pretrained(e5_ort_model)
e5_qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)

e5_int8_dir = OUTPUT_DIR / "e5_int8"
e5_quantizer.quantize(save_dir=str(e5_int8_dir), quantization_config=e5_qconfig)
e5_tokenizer.save_pretrained(str(e5_int8_dir))

In [ ]:
import faiss

scripts_index_rows = load_jsonl(DATA_DIR / "scripts_index.jsonl")
print(f"scripts_index rows: {len(scripts_index_rows)}")

e5_embed_model = ORTModelForFeatureExtraction.from_pretrained(e5_int8_dir)
e5_embed_tokenizer = AutoTokenizer.from_pretrained(e5_int8_dir)


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(1)
    counts = mask.sum(1).clamp(min=1e-9)
    return summed / counts


def embed_passages(texts, batch_size=64):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = ["passage: " + t[:2000] for t in texts[i:i + batch_size]]
        enc = e5_embed_tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt")
        with torch.no_grad():
            out = e5_embed_model(**enc)
        pooled = mean_pool(out.last_hidden_state, enc["attention_mask"])
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_vecs.append(pooled.numpy())
    return np.concatenate(all_vecs, axis=0)


script_embeddings = embed_passages([r["text"] for r in scripts_index_rows])
print(script_embeddings.shape)

In [ ]:
faiss_index = faiss.IndexFlatIP(script_embeddings.shape[1])
faiss_index.add(script_embeddings.astype(np.float32))
faiss.write_index(faiss_index, str(OUTPUT_DIR / "faiss.index"))

scripts_meta = [
    {"script_id": r["script_id"], "family": r["family"], "excerpt": r["text"][:120]}
    for r in scripts_index_rows
]
with open(OUTPUT_DIR / "scripts_meta.json", "w", encoding="utf-8") as f:
    json.dump(scripts_meta, f, ensure_ascii=False, indent=2)

print(f"faiss index: {faiss_index.ntotal} vectors")

## Summary

Everything under `OUTPUT_DIR` is what the HF Space image needs: both INT8 classifiers, the INT8 embedder, the FAISS index + metadata, calibration, and both confusion matrix PNGs for the README/model card.

In [ ]:
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(OUTPUT_DIR)}  ({p.stat().st_size / 1024:.1f} KB)")